In [1]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np

X, y = make_classification(
    n_samples=100_000,
    n_features=25,
    n_informative=12,
    n_redundant=5,
    weights=[0.99, 0.01],
    flip_y=0.01,
    random_state=42
)
print(f"Fraud cases : {y.sum()}")
print(f"Fraud rate  : {y.mean()*100:.3f}%")

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Scale for fair comparison — trees don't need scaling
# but having it here makes the pipeline consistent
scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_train)
X_v_s  = scaler.transform(X_val)

# Compute scale_pos_weight for XGBoost
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
spw = neg / pos
print(f"scale_pos_weight : {spw:.1f}")

Fraud cases : 1497
Fraud rate  : 1.497%
scale_pos_weight : 65.8


In [2]:
def fraud_report(y_true, y_pred, y_proba, label="Model"):
    from sklearn.metrics import (
        classification_report, roc_auc_score,
        average_precision_score, confusion_matrix
    )
    print(f"\n{'='*55}")
    print(f"  {label}")
    print(f"{'='*55}")
    print(classification_report(y_true, y_pred, digits=4))
    print(f"ROC-AUC  : {roc_auc_score(y_true, y_proba):.4f}")
    print(f"PR-AUC   : {average_precision_score(y_true, y_proba):.4f}")
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    print(f"TP={tp:5d} | FP={fp:5d} | FN={fn:5d} | TN={tn:7d}")
    print(f"Catch Rate (Recall) : {tp/(tp+fn):.4f}")
    print(f"False Alarm Rate    : {fp/(fp+tn):.6f}")

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score


"""
this is simple RF model trained with basic paramters
"""
rf = RandomForestClassifier(
    n_estimators=300,       # 300 trees — enough for stability
    max_depth=10,           # moderate depth per tree
    min_samples_leaf=20,    # prevents micro-leaves
    max_features='sqrt',    # key randomness lever
    class_weight='balanced',
    n_jobs=-1,
    random_state=42
)
rf.fit(X_tr_s, y_train)

y_pred_rf  = rf.predict(X_v_s)
y_proba_rf = rf.predict_proba(X_v_s)[:, 1]

fraud_report(y_val, y_pred_rf, y_proba_rf, "Random Forest")

# Check if more trees help — plot OOB error
# Out-of-bag error: each tree evaluated on samples it did not see
rf_oob = RandomForestClassifier(
    n_estimators=500,
    max_depth=10,
    min_samples_leaf=20,
    max_features='sqrt',
    class_weight='balanced',
    oob_score=True,         # enables out-of-bag evaluation
    n_jobs=-1,
    random_state=42
)
rf_oob.fit(X_tr_s, y_train)
print(f"OOB Score: {rf_oob.oob_score_:.4f}")
# OOB score gives a free validation estimate without a separate val set

"""

so this out of box validation just evaluates the model on unseen train sample without separte test data imagine like this each tree has random samples to rain
so some samples are not passed to lets say tree x and to that tree we pass those unseen samples to evalaute performanve

"""
# Convergence plot — does adding more trees still help?
import matplotlib.pyplot as plt

pr_auc_by_trees = []
for n_trees in range(10, 310, 10):
    rf_temp = RandomForestClassifier(
        n_estimators=n_trees,
        max_depth=10,
        min_samples_leaf=20,
        max_features='sqrt',
        class_weight='balanced',
        n_jobs=-1,
        random_state=42
    )
    rf_temp.fit(X_tr_s, y_train)
    pr = average_precision_score(
        y_val, rf_temp.predict_proba(X_v_s)[:,1])
    pr_auc_by_trees.append(pr)

    """

it helps us understand how much adding more trees will help and when adding more trees become useless like in validation curve we use to check optjimal 
hyperparrameter by checking the best test sweet spot so here at differnt no of trees we train and each time we check PR AUC then plot on x axis no of trees
and y axis PR AUC to check either adding more tree still helps


    """

plt.figure(figsize=(10, 4))
plt.plot(range(10, 310, 10), pr_auc_by_trees)
plt.xlabel('Number of trees')
plt.ylabel('PR-AUC')
plt.title('Random Forest convergence — when do more trees stop helping?')
plt.tight_layout()
plt.show()
# The point where this curve flattens = your optimal n_estimators


  Random Forest
              precision    recall  f1-score   support

           0     0.9923    0.9910    0.9916     19701
           1     0.4537    0.4916    0.4719       299

    accuracy                         0.9836     20000
   macro avg     0.7230    0.7413    0.7318     20000
weighted avg     0.9842    0.9836    0.9839     20000

ROC-AUC  : 0.8225
PR-AUC   : 0.4762
TP=  147 | FP=  177 | FN=  152 | TN=  19524
Catch Rate (Recall) : 0.4916
False Alarm Rate    : 0.008984
OOB Score: 0.9833


In [ ]:
#-------------XGBOOST----------------

In [ ]:
import xgboost as xgb

"""
this is XGboost with differnet parameters mainly important 


"""
xgb_model = xgb.XGBClassifier(
    n_estimators=1000,          # set high — early stopping will halt
    learning_rate=0.05,         # small steps, more iterations
    max_depth=4,                # shallow trees for fraud
    min_child_weight=10,        # min sum of instance weight in leaf
    subsample=0.8,              # row subsampling per tree
    colsample_bytree=0.8,       # feature subsampling per tree
    gamma=1,                    # min loss reduction to make a split
    reg_alpha=0.1,              # L1 regularisation
    reg_lambda=1.0,             # L2 regularisation
    scale_pos_weight=spw,       # handles class imbalance
    eval_metric='aucpr',        # PR-AUC as eval metric
    early_stopping_rounds=30,   # stop if no improvement for 30 rounds
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(
    X_tr_s, y_train,
    eval_set=[(X_tr_s, y_train), (X_v_s, y_val)],
    verbose=50          # print every 50 rounds
)

print(f"\nBest iteration : {xgb_model.best_iteration}")
print(f"Best val PR-AUC: {xgb_model.best_score:.4f}")

y_pred_xgb  = xgb_model.predict(X_v_s)
y_proba_xgb = xgb_model.predict_proba(X_v_s)[:, 1]

fraud_report(y_val, y_pred_xgb, y_proba_xgb, "XGBoost")

In [ ]:
#---------------LGBM it actually splits not lvl by lvl but from 1 side it is good for large data faster but for small overift
#so we basically increase this leaf

In [ ]:
import lightgbm as lgb

lgb_model = lgb.LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=31,              # primary complexity control
    min_child_samples=50,       # equivalent to min_samples_leaf
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    is_unbalance=True,          # handles class imbalance
    random_state=42,
    n_jobs=-1,
    verbosity=-1                # suppress verbose output
)

callbacks = [lgb.early_stopping(30), lgb.log_evaluation(50)]

lgb_model.fit(
    X_tr_s, y_train,
    eval_set=[(X_v_s, y_val)],
    eval_metric='average_precision',
    callbacks=callbacks
)

print(f"Best iteration : {lgb_model.best_iteration_}")

y_pred_lgb  = lgb_model.predict(X_v_s)
y_proba_lgb = lgb_model.predict_proba(X_v_s)[:, 1]

fraud_report(y_val, y_pred_lgb, y_proba_lgb, "LightGBM")